# Setup

In [1]:
import numpy as np
import pandas as pd

In [2]:
def shuffle_along_axis(arr, axis):
    idx = np.random.rand(*arr.shape).argsort(axis=axis)
    return np.take_along_axis(arr, idx, axis=axis)


def shuffled(arr):
    arr_shuffled = arr.copy()
    np.random.shuffle(arr_shuffled)
    return arr_shuffled


def choose_n_and_delete(arr, N):
    chosen = np.random.choice(arr, size=N, replace=False)
    arr = np.delete(arr, np.where(np.isin(arr, chosen)))
    return chosen, arr

## Settings

In [4]:
DIR_NAME = "/Users/ccnlab/Development/sequences/shaping/v1_rlwm"

exp_type = "v1_rlwm"
num_conditions = 2  # number of different file sequences to generate (change if needed)
exact_reps = True  # should the number of repetitions be exact or ok to exceed by 1?

block_structures = (
    np.array([1, 2, 3]),
    np.array([1, 3, 2]),
    np.array([2, 2, 2]),
    np.array([3, 2, 1]),
)

# Repeat 4 times (by stacking), then shuffle rows
repeated_blocks = np.vstack([block_structures for _ in range(4)])
np.random.shuffle(repeated_blocks)
SET_SIZE_TO_KEY_LIST = {
    2: np.vstack(
        (
            np.array([1, 0, 1]),
            np.array([1, 1, 0]),
            np.array([0, 1, 1]),
            np.array([1, 0, 1]),
        )
    ),
    6: repeated_blocks,
    12: repeated_blocks,
}

# RLWM seq generation

In [5]:
from pathlib import Path

from tqdm import trange

COLNAMES = [
    "stim",
    "correct_key",
    "set_size",
    "block",
    "img_folder",
    "condition",
]


class RLWMSequenceMaker:
    def __init__(
        self,
        exp_type,
        num_reps,
        block_structure,
        num_conditions=2,
        exact_reps=False,
        to_dir=None,
        num_keys=3,
        max_stims=6,
    ):
        self.exp_type = exp_type
        self.num_reps = num_reps
        self.num_conditions = num_conditions
        self.exact_reps = exact_reps
        self.num_keys = num_keys
        self.max_stims = max_stims
        self.block_structure = list(block_structure)
        self.num_blocks = len(self.block_structure)
        self.set_size_to_key_list = SET_SIZE_TO_KEY_LIST

    def make_sequences(self):
        """Generate one CSV / DataFrame per seq. Returns all DataFrame."""

        blocks = list(self.block_structure)
        block_rules = self._make_block_rules(blocks)
        stim_sets = np.random.permutation(self.num_blocks) + 2

        block_dfs = []
        for block_i, (ns, condition) in enumerate(blocks):
            prototype = self._make_seq_prototype(ns)
            correct_keys = np.vectorize(block_rules[block_i].get)(prototype - 1)

            block_dfs.append(
                pd.DataFrame(
                    {
                        "stim": prototype,
                        "correct_key": correct_keys,
                        "set_size": ns,
                        "block": block_i + 1,
                        "img_folder": stim_sets[block_i],
                        "condition": condition,
                    }
                )
            )

        return pd.concat(block_dfs, ignore_index=True)[COLNAMES]

    def _make_block_rules(self, blocks):
        """stim_idx (0-based) -> correct key, per block."""
        key_pools = {
            ns: shuffle_along_axis(keys, axis=1).tolist()
            for ns, keys in self.set_size_to_key_list.items()
        }

        block_rules = []
        for ns, _ in blocks:
            counts = key_pools[ns].pop()
            keys_for_stims = [
                key_i for key_i in range(self.num_keys) for _ in range(counts[key_i])
            ]
            block_rules.append(dict(enumerate(keys_for_stims)))
        return block_rules

    def _make_block_stimuli(self, blocks):
        """stim_idx (0-based) -> image number (1-based), per block."""
        return [
            dict(enumerate((np.random.permutation(self.max_stims) + 1)[:ns]))
            for ns, _ in blocks
        ]

    def _make_seq_prototype(self, set_size):
        """Stimulus indices 1..set_size, reshuffled each repetition cycle."""
        cycles = [
            shuffled(np.arange(1, set_size + 1)) for _ in range(self.num_reps + 1)
        ]
        return np.hstack(cycles)

In [8]:
first_block_structure = [(12, 0)] + [(12, 1)]
second_block_structure = [(12, 1)] + [(12, 0)]
for idx, bs in enumerate([first_block_structure, second_block_structure]):
    seqmkr = RLWMSequenceMaker(
        exp_type=exp_type,
        num_reps=7,
        block_structure=bs,
        num_conditions=2,
        exact_reps=exact_reps,
    )
    train_df = seqmkr.make_sequences()

    out_path = f"{DIR_NAME}/seq{idx+1}_learning.csv"
    train_df.to_csv(out_path, index=False)

# Checks

In [9]:
for i in np.arange(1, 2):
    df = pd.read_csv(f"{DIR_NAME}/seq{i}_learning.csv")

    print(df.groupby(["block"]).size())

block
1    96
2    96
dtype: int64
